In [154]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('imports successful...')

imports successful...


In [155]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [156]:
df.isna().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [157]:
df_base = df[['engine_displacement',
'horsepower',
'vehicle_weight',
'model_year']]

In [158]:
df_base.head()

,engine_displacement,horsepower,vehicle_weight,model_year
0,2180,243.0,3870,2006
1,2390,272.0,4210,2008
2,2320,267.0,4240,1996
3,2130,258.0,4490,1989
4,2580,304.0,4510,1994


In [159]:
df_base.horsepower.describe()

count    9123.000000
mean      254.446016
std        22.844613
min       171.000000
25%       239.000000
50%       254.000000
75%       270.000000
max       334.000000
Name: horsepower, dtype: float64

In [160]:
def split_y_data(df_train, df_val, df_test):
    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values

    return y_train, y_val, y_test

In [161]:
def split_X_data(df, seed_val=42): 
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed_val)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    return df_train, df_val, df_test

In [162]:
base = ['engine_displacement','horsepower','vehicle_weight','model_year']


In [163]:
df_train, df_val, df_test = split_X_data(df)

In [164]:
y_train, y_val, y_test = split_y_data(df_train, df_val, df_test)

In [165]:
df_train_zeros = df_train.copy()
df_train_mean = df_train.copy()

In [166]:
hp_mean = df_train_mean.horsepower.mean().round(4)
print(hp_mean)

254.4634


In [167]:
df_train_mean.horsepower.mean()

np.float64(254.46338337605272)

In [168]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [169]:
def linear_regression(xi):
    n - len(xi)
    pred = w0

    for j in range(n):
        pred += w[j] * xi[j]
    
    return pred

In [170]:
X_train = df_train[base].values

In [171]:
train_linear_regression(X_train, y_train)

(np.float64(nan), array([nan, nan, nan, nan]))

In [172]:
df_train[base].isnull().sum()

engine_displacement      0
horsepower             538
vehicle_weight           0
model_year               0
dtype: int64

In [173]:
def prepare(df, val=0):
  df = df.copy()
  df_num = df[base]
  df_num = df_num.fillna(val)
  X = df_num.values
  return X

In [174]:
X_train_zeros = prepare(df_train_zeros)
X_train_mean = prepare(df_train_mean, hp_mean)

In [175]:
w0_z, w_z = train_linear_regression(X_train_zeros, y_train)

In [176]:
w0_m, w_m = train_linear_regression(X_train_mean, y_train)

In [177]:
X_val = prepare(df_val)

In [178]:
# using zero fills
y_pred_z = w0_z + X_val.dot(w_z)

In [179]:
# using mean fills
y_pred_m = w0_m + X_val.dot(w_m)

In [180]:
def rmse(y, y_pred):
    err = y - y_pred
    sqr_err = err**2
    mse = sqr_err.mean()

    return np.sqrt(mse)

In [181]:
# zeros run
rmse(y_val, y_pred_z).round(3)

np.float64(2.205)

In [182]:
# mean run
rmse(y_val, y_pred_m).round(3)

np.float64(2.232)

In [183]:
def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [184]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)

    X_val = prepare(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    print(r, w0, score.round(4))


0 -153.88496188223104 2.2053
0.01 -148.3092124404723 2.2058
0.1 -111.83871039307081 2.2241
1 -32.331865964969374 2.3492
5 -7.772852209978469 2.4094
10 -3.987116416019025 2.4195
100 -0.4082196488463344 2.4292


In [185]:
def strip_base(df_train, df_val, df_test):
    return df_train[base], df_val[base], df_test[base]

In [186]:
def evaluate(df_val, y):
    X_val = prepare(df_val)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y, y_pred)
    
    print(seed, w0, score.round(4))
    return score

In [187]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
score_list = []

for seed in seeds:
    df_train, df_val, df_test = split_X_data(df, seed_val=seed)
    y_train, y_val, y_test = split_y_data(df_train, df_val, df_test)
    df_train, df_val, df_test = strip_base(df_train, df_val, df_test)

    # fill missing values
    X_train = prepare(df_train)
    w0, w = train_linear_regression(X_train, y_train)
    
    score_list.append(evaluate(df_val, y_val))

np.std(score_list).round(3)

0 -150.42885945053604 2.2392
1 -151.61231559524666 2.2021
2 -152.84068112584038 2.1634
3 -152.33298096679223 2.2044
4 -158.88856816457883 2.2019
5 -151.8573533887713 2.2458
6 -154.3717913727048 2.2697
7 -155.0183525899584 2.1908
8 -150.4381740048234 2.2166
9 -154.36007488958072 2.207


np.float64(0.029)

In [188]:
df_train, df_val, df_test = split_X_data(df, seed_val=9)
y_train, y_val, y_test = split_y_data(df_train, df_val, df_test)
df_train, df_val, df_test = strip_base(df_train, df_val, df_test)

# fill missing values
full_train = pd.concat([df_train, df_val])
y_full_train = np.concatenate([y_train, y_val])

X_train = prepare(full_train)
w0, w = train_linear_regression_reg(X_train, y_full_train, r=0.001)

evaluate(df_test, y_test)


9 -152.37535772493302 2.2358


np.float64(2.2358277471917636)